# Business Concept Synonym Setup

Attach this Fabric notebook to the target non-schema-enabled Lakehouse before running it. Each Spark SQL operation is intentionally isolated in its own cell because the Fabric SQL interpreter can reject multiple top-level statements submitted in one cell.


## 1. Create the Delta Synonym Table

Run this cell separately to create the managed Delta table in the attached Lakehouse.


In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS business_concept_synonyms (
  BusinessConceptId STRING NOT NULL,
  ConceptGroupId STRING NOT NULL,
  CanonicalName STRING NOT NULL,
  Synonyms STRING NOT NULL,
  Description STRING,
  IsActive BOOLEAN NOT NULL
) USING DELTA


## 2. Build the Source Dataset

Create a DataFrame with an explicit schema and register it as the temporary view `business_concept_synonyms_source`. The next SQL cell uses this view as the `MERGE` source.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import BooleanType, StringType, StructField, StructType

spark_session = SparkSession.getActiveSession()
if spark_session is None:
    raise RuntimeError("No active Spark session. Run this notebook in Fabric with a Lakehouse attached.")

synonym_schema = StructType([
    StructField("BusinessConceptId", StringType(), nullable=False),
    StructField("ConceptGroupId", StringType(), nullable=False),
    StructField("CanonicalName", StringType(), nullable=False),
    StructField("Synonyms", StringType(), nullable=False),
    StructField("Description", StringType(), nullable=True),
    StructField("IsActive", BooleanType(), nullable=False),
])

synonym_records = [
    ("bc-gmv-001", "bc-gmv", "GrossMerchandiseValue", "GrossMerchandiseValue", "Total value of merchandise sold before deductions.", True),
    ("bc-gmv-002", "bc-gmv", "GrossMerchandiseValue", "GMV", "Total value of merchandise sold before deductions.", True),
    ("bc-gmv-003", "bc-gmv", "GrossMerchandiseValue", "gross sales", "Total value of merchandise sold before deductions.", True),
    ("bc-gmv-004", "bc-gmv", "GrossMerchandiseValue", "Velvet Compass", "Opaque private code phrase for gross merchandise value.", True),
    ("bc-customer-001", "bc-customer", "Customer", "Customer", "A person or organization purchasing products or services.", True),
    ("bc-customer-002", "bc-customer", "Customer", "cust", "A person or organization purchasing products or services.", True),
    ("bc-customer-003", "bc-customer", "Customer", "client", "A person or organization purchasing products or services.", True),
    ("bc-customer-004", "bc-customer", "Customer", "Marble Lantern", "Opaque private code phrase for a customer.", True),
    ("bc-sales-order-001", "bc-sales-order", "SalesOrder", "SalesOrder", "A confirmed customer purchase represented in the ontology.", True),
    ("bc-sales-order-002", "bc-sales-order", "SalesOrder", "sales order", "A confirmed customer purchase represented in the ontology.", True),
    ("bc-sales-order-003", "bc-sales-order", "SalesOrder", "booking", "A confirmed customer purchase represented in the ontology.", True),
    ("bc-sales-order-004", "bc-sales-order", "SalesOrder", "bookings", "A confirmed customer purchase represented in the ontology.", True),
    ("bc-net-revenue-001", "bc-net-revenue", "NetRevenue", "NetRevenue", "Revenue after returns, discounts, and allowances.", True),
    ("bc-net-revenue-002", "bc-net-revenue", "NetRevenue", "net rev", "Revenue after returns, discounts, and allowances.", True),
    ("bc-net-revenue-003", "bc-net-revenue", "NetRevenue", "net sales", "Revenue after returns, discounts, and allowances.", True),
    ("bc-net-revenue-004", "bc-net-revenue", "NetRevenue", "Amber Kite", "Opaque private code phrase for net revenue.", True),
    ("bc-retired-001", "bc-retired", "LegacySalesMetric", "old sales KPI", "Retired demonstration mapping.", False),
]

synonym_source_df = spark_session.createDataFrame(synonym_records, schema=synonym_schema)
synonym_source_df.createOrReplaceTempView("business_concept_synonyms_source")
display(synonym_source_df.orderBy("ConceptGroupId", "BusinessConceptId"))


## 3. Upsert Synonyms with Delta MERGE

Run the `MERGE` as a separate Spark SQL statement. Explicit column mappings keep the upsert behavior stable if the table or source schema later changes.


In [ ]:
%%sql
MERGE INTO business_concept_synonyms AS target
USING business_concept_synonyms_source AS source
ON target.BusinessConceptId = source.BusinessConceptId
WHEN MATCHED THEN UPDATE SET
  target.ConceptGroupId = source.ConceptGroupId,
  target.CanonicalName = source.CanonicalName,
  target.Synonyms = source.Synonyms,
  target.Description = source.Description,
  target.IsActive = source.IsActive
WHEN NOT MATCHED THEN INSERT (
  BusinessConceptId,
  ConceptGroupId,
  CanonicalName,
  Synonyms,
  Description,
  IsActive
) VALUES (
  source.BusinessConceptId,
  source.ConceptGroupId,
  source.CanonicalName,
  source.Synonyms,
  source.Description,
  source.IsActive
)


## 4. Inspect the Seeded Records

Review all rows after the idempotent upsert and verify both active and retired mappings.


In [ ]:
%%sql
SELECT
  BusinessConceptId,
  ConceptGroupId,
  CanonicalName,
  Synonyms,
  Description,
  IsActive
FROM business_concept_synonyms
ORDER BY ConceptGroupId, BusinessConceptId


## 5. Run the Synonym Ambiguity Quality Gate

This query must return zero rows. Any result means one normalized active synonym maps to multiple canonical names.


In [ ]:
%%sql
SELECT
  lower(trim(Synonyms)) AS NormalizedSynonym,
  count(DISTINCT CanonicalName) AS CanonicalCount,
  sort_array(collect_set(CanonicalName)) AS CanonicalNames
FROM business_concept_synonyms
WHERE IsActive = true
GROUP BY lower(trim(Synonyms))
HAVING count(DISTINCT CanonicalName) > 1
ORDER BY NormalizedSynonym


## 6. Assert the Quality-Gate Result

Repeat the quality query in PySpark so invalid mappings fail interactive runs and pipeline executions instead of producing only a visible result set.


In [ ]:
from pyspark.sql import SparkSession

spark_session = SparkSession.getActiveSession()
if spark_session is None:
    raise RuntimeError("No active Spark session. Run this notebook in Fabric with a Lakehouse attached.")

ambiguity_df = spark_session.sql("""
    SELECT
      lower(trim(Synonyms)) AS NormalizedSynonym,
      count(DISTINCT CanonicalName) AS CanonicalCount,
      sort_array(collect_set(CanonicalName)) AS CanonicalNames
    FROM business_concept_synonyms
    WHERE IsActive = true
    GROUP BY lower(trim(Synonyms))
    HAVING count(DISTINCT CanonicalName) > 1
    ORDER BY NormalizedSynonym
""")

if ambiguity_df.limit(1).count() > 0:
    display(ambiguity_df)
    raise ValueError("Synonym ambiguity quality gate failed: an active synonym maps to multiple canonical names.")

print("PASS: every active normalized synonym maps to at most one canonical name.")


## 7. Create the Demo Business Data

Create two small managed Delta tables that make the synonym behavior visible in the ontology. These tables are owned by the sample and are replaced on each run.


In [ ]:
from datetime import date

from pyspark.sql.types import DateType, DoubleType

customer_schema = StructType([
    StructField("CustomerId", StringType(), nullable=False),
    StructField("CustomerName", StringType(), nullable=False),
    StructField("Region", StringType(), nullable=False),
])
customer_records = [
    ("C001", "Contoso Retail", "North America"),
    ("C002", "Alpine Stores", "Europe"),
    ("C003", "Fabrikam Market", "Asia Pacific"),
]

sales_order_schema = StructType([
    StructField("SalesOrderId", StringType(), nullable=False),
    StructField("CustomerId", StringType(), nullable=False),
    StructField("OrderDate", DateType(), nullable=False),
    StructField("OrderStatus", StringType(), nullable=False),
    StructField("GrossMerchandiseValue", DoubleType(), nullable=False),
    StructField("NetRevenue", DoubleType(), nullable=False),
])
sales_order_records = [
    ("SO-1001", "C001", date(2026, 8, 3), "Completed", 12500.0, 11800.0),
    ("SO-1002", "C001", date(2026, 8, 18), "Completed", 8200.0, 7900.0),
    ("SO-1003", "C002", date(2026, 8, 9), "Returned", 5100.0, 0.0),
    ("SO-1004", "C002", date(2026, 9, 2), "Completed", 9700.0, 9300.0),
    ("SO-1005", "C003", date(2026, 9, 12), "Pending", 4300.0, 4100.0),
]

customers_df = spark_session.createDataFrame(customer_records, schema=customer_schema)
sales_orders_df = spark_session.createDataFrame(sales_order_records, schema=sales_order_schema)

table_frames = {
    "customers": customers_df,
    "sales_orders": sales_orders_df,
}
required_columns = {
    "customers": {"CustomerId", "CustomerName", "Region"},
    "sales_orders": {
        "SalesOrderId",
        "CustomerId",
        "OrderDate",
        "OrderStatus",
        "GrossMerchandiseValue",
        "NetRevenue",
    },
}

for table_name, table_df in table_frames.items():
    spark_session.sql(f"DROP TABLE IF EXISTS {table_name}")
    table_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    if not spark_session.catalog.tableExists(table_name):
        raise RuntimeError(f"Managed Delta table was not registered: {table_name}")
    actual_columns = set(spark_session.table(table_name).columns)
    missing_columns = required_columns[table_name] - actual_columns
    if missing_columns:
        raise RuntimeError(f"{table_name} is missing columns: {sorted(missing_columns)}")

print(f"PASS: created managed Delta tables {', '.join(table_frames)}")
print("The ontology bindings use the unqualified table names and omit sourceSchema.")
display(customers_df.orderBy("CustomerId"))
display(sales_orders_df.orderBy("SalesOrderId"))


## 8. Validate Graph Source Readiness

Verify that all ontology sources are managed Delta tables without column mapping, entity keys are unique and non-null, and every relationship row references an existing customer. Run this audit before refreshing the ontology's managed Graph child item.


In [ ]:
from pyspark.sql import functions as F

source_keys = {
    "business_concept_synonyms": "BusinessConceptId",
    "customers": "CustomerId",
    "sales_orders": "SalesOrderId",
}

for table_name, key_column in source_keys.items():
    detail = spark_session.sql(f"DESCRIBE DETAIL {table_name}").first().asDict(recursive=True)
    if str(detail.get("format", "")).lower() != "delta":
        raise ValueError(f"{table_name} must be a Delta table.")

    location = str(detail.get("location", "")).replace("\\", "/")
    if "/Tables/" not in location:
        raise ValueError(f"{table_name} is not a managed Lakehouse table: {location}")

    properties = detail.get("properties") or {}
    column_mapping_mode = str(properties.get("delta.columnMapping.mode", "none")).lower()
    if column_mapping_mode not in {"", "none"}:
        raise ValueError(f"{table_name} has unsupported Delta column mapping: {column_mapping_mode}")

    table_df = spark_session.table(table_name)
    invalid_key_count = table_df.filter(F.col(key_column).isNull()).count()
    duplicate_key_count = (
        table_df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    if invalid_key_count or duplicate_key_count:
        raise ValueError(
            f"{table_name}.{key_column} has {invalid_key_count} null and "
            f"{duplicate_key_count} duplicate key groups."
        )

orphan_customer_count = (
    spark_session.table("sales_orders")
    .select("CustomerId")
    .distinct()
    .join(
        spark_session.table("customers").select("CustomerId").distinct(),
        on="CustomerId",
        how="left_anti",
    )
    .count()
)
if orphan_customer_count:
    raise ValueError(f"sales_orders contains {orphan_customer_count} unknown CustomerId values.")

print("PASS: graph sources are managed Delta tables without column mapping.")
print("PASS: entity keys are non-null and unique.")
print("PASS: every SalesOrder relationship row references an existing Customer.")
